In [ ]:
!pip install -q 'transformers>=4.40.0'
!pip install -q datasets evaluate accelerate albumentations roboflow

In [ ]:
import os, cv2, torch
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
import evaluate

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if result.returncode == 0:
    print(result.stdout)
    print(' GPU is available! You are good to go.')
else:
    print(' No GPU detected!')
    print(' Go to Runtime → Change Runtime Type → T4 GPU, then re-run.')
print(f' Using device: {device}')
if device.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
from roboflow import Roboflow

USE_DEMO_DATA = True
API_KEY = 'YOUR_API_KEY_HERE'
WORKSPACE = 'YOUR_WORKSPACE_HERE'
PROJECT_NAME = 'YOUR_PROJECT_NAME_HERE'
VERSION = 1

In [ ]:
ID2LABEL = {0: 'background', 1: 'object'}
LABEL2ID = {v: k for k, v in ID2LABEL.items()}
NUM_CLASSES = len(ID2LABEL)

In [ ]:
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
version = project.version(ROBOFLOW_VERSION)
dataset = version.download("png-mask-semantic")
DATA_BASE      = dataset.location

TRAIN_IMG_DIR  = os.path.join(DATA_BASE, 'train')
TRAIN_MASK_DIR = os.path.join(DATA_BASE, 'train')
VALID_IMG_DIR  = os.path.join(DATA_BASE, 'valid')
VALID_MASK_DIR = os.path.join(DATA_BASE, 'valid')

In [ ]:
def preview_samples(img_dir, mask_dir, n=4):
    all_files = sorted(os.listdir(img_dir))
    image_files = [f for f in all_files if f.lower().endswith(('.jpg', '.jpeg', '.png')) and '_mask.png' not in f][:n]

    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1: axes = [axes]

    for i, fname in enumerate(image_files):
        img_path = os.path.join(img_dir, fname)
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)

        base_name = os.path.splitext(fname)[0]
        mask_path = os.path.join(mask_dir, f"{base_name}_mask.png")

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is None:
            print(f' Mask not found: {mask_path}'); continue

        overlay = img.copy()
        overlay[mask > 0] = (overlay[mask > 0] * 0.5 + np.array([255,0,0]) * 0.5).astype(np.uint8)

        axes[i][0].imshow(img);                 axes[i][0].set_title('Image');   axes[i][0].axis('off')
        axes[i][1].imshow(mask, cmap='gray');   axes[i][1].set_title('Mask');    axes[i][1].axis('off')
        axes[i][2].imshow(overlay);             axes[i][2].set_title('Overlay'); axes[i][2].axis('off')

    plt.suptitle('Dataset Preview', fontsize=14, y=1.01)
    plt.tight_layout(); plt.show()

    if image_files:
        test_img = cv2.imread(os.path.join(img_dir, image_files[0]))
        print(f'Image shape: {test_img.shape}')
        print(f'Mask unique values: {np.unique(mask).tolist()}')

preview_samples(TRAIN_IMG_DIR, TRAIN_MASK_DIR, n=4)

In [ ]:
class SegmentationDataset(Dataset):
  def __init__(self, img_dir, mask_dir, processor):
    self.img_dir = img_dir
    self.mask_dir = mask_dir
    self.processor = processor
    self.images = sorted([
        f for f in os.listdir(img_dir)
        if f.lower().endswith(('.png', '.jpg', '.jpeg')) and '_mask.png' not in f
    ])

  def __len__(self):
    return len(self.images)

  def __getitem__(self, idx):
    fname = self.images[idx]
    image_path = os.path.join(self.img_dir, fname)
    image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)

    base_name = os.path.splitext(fname)[0]
    mask_path = os.path.join(self.mask_dir, f"{base_name}_mask.png")

    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if mask is None:
      raise FileNotFoundError(f"Mask not found for: {mask_path}")

    mask = (mask > 0).astype(np.uint8)

    encoded = self.processor(
        images = image,
        segmentation_maps = mask,
        return_tensors = 'pt'
    )
    return {
        'pixel_values': encoded['pixel_values'].squeeze(0),
        'labels'      : encoded['labels'].squeeze(0).long(),
      }

In [ ]:
processor = SegformerImageProcessor(do_reduce_labels=False)

train_dataset = SegmentationDataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR, processor)
valid_dataset = SegmentationDataset(VALID_IMG_DIR, VALID_MASK_DIR, processor)

print(f'Training samples  : {len(train_dataset)}')
print(f'Validation samples: {len(valid_dataset)}')

sample = train_dataset[0]
print(f'\nSample pixel_values shape: {sample["pixel_values"].shape}  (channels, height, width)')
print(f'Sample labels shape      : {sample["labels"].shape}  (height, width)')
print(f'Unique label values      : {sample["labels"].unique().tolist()}  (should be 0 and 1)')

In [ ]:
BATCH_SIZE = 4

train_dataloader = DataLoader(
    train_dataset,
    batch_size = BATCH_SIZE,
    shuffle    = True,
    num_workers= 2,
    pin_memory = True
)

valid_dataloader = DataLoader(
    valid_dataset,
    batch_size = BATCH_SIZE,
    shuffle    = False,
    num_workers= 2,
    pin_memory = True
)

| Model | Params | For |
|-------|--------|-----|
| **B0** | 3.7M |  Free Colab, learning |
| B1 | 13.7M | Better accuracy |
| B2 | 24.7M | Good balance |
| B3 | 44.6M | High accuracy |
| B5 | 81.4M | Best (needs Colab Pro) |

In [ ]:
model = SegformerForSemanticSegmentation.from_pretrained(
    'nvidia/segformer-b0-finetuned-ade-512-512',
    num_labels = NUM_CLASSES,
    id2label = ID2LABEL,
    label2id = LABEL2ID,
    ignore_mismatched_sizes = True
)

model = model.to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f' SegFormer-B0 loaded')
print(f'   Total parameters    : {total:,}')
print(f'   Trainable parameters: {trainable:,}')
print(f'   Running on          : {next(model.parameters()).device}')

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 10

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr = 6e-5,
    weight_decay = 0.01
)

scheduler = CosineAnnealingLR(
    optimizer,
    T_max = EPOCHS,
)

use_amp = (device.type == 'cuda')
scaler = GradScaler(enabled=use_amp)
metric = evaluate.load('mean_iou')

In [ ]:
history = {'train_loss': [], 'val_miou': []}
best_miou = 0.0
best_model_path = '/content/best_segformer.pt'

print('Starting training...\n')
print(f'{"Epoch":>6}  {"Train Loss":>12}  {"Val mIoU":>10}  {"LR":>10}')
print('-' * 46)

for epoch in range(1, EPOCHS + 1):
  model.train()
  epoch_loss = 0.0
  for batch in train_dataloader:
    pixel_values = batch['pixel_values'].to(device)
    labels = batch['labels'].to(device)

    optimizer.zero_grad()
    with autocast(enabled=use_amp):
      outputs = model(pixel_values=pixel_values, labels=labels)
      loss = outputs.loss

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    epoch_loss += loss.item()

  avg_loss = epoch_loss / len(train_dataloader)
  history['train_loss'].append(avg_loss)

  model.eval()
  with torch.no_grad():
    for batch in valid_dataloader:
      pixel_values = batch['pixel_values'].to(device)
      labels = batch['labels'].to(device)

      outputs = model(pixel_values=pixel_values)
      logits = torch.nn.functional.interpolate(
          outputs.logits,
          size = labels.shape[-2:],
          mode = 'bilinear',
          align_corners = False
      )
      predicted = logits.argmax(dim=1)
      metric.add_batch(
          predictions = predicted.detach().cpu().numpy(),
          references = labels.detach().cpu().numpy()
      )

  results = metric.compute(num_labels=NUM_CLASSES, ignore_index=255)
  miou = results['mean_iou']
  history['val_miou'].append(miou)

  saved = ''
  if miou > best_miou:
    best_miou = miou
    torch.save(model.state_dict(), best_model_path)
    saved = ' (saved)'

  lr = scheduler.get_last_lr()[0]
  print(f'{epoch:>6}  {avg_loss:>12.4f}  {miou:>10.4f}  {lr:>10.6f}{saved}')
  scheduler.step()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = range(1, EPOCHS + 1)

axes[0].plot(epochs_range, history['train_loss'], 'b-o', linewidth=2, markersize=5)
axes[0].set_title('Training Loss', fontsize=14)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['val_miou'], 'g-o', linewidth=2, markersize=5)
best_ep = history['val_miou'].index(max(history['val_miou'])) + 1
axes[1].axvline(x=best_ep, color='red', linestyle='--', alpha=0.7, label=f'Best (epoch {best_ep})')
axes[1].set_title('Validation mIoU', fontsize=14)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mIoU'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('SegFormer Training Results', fontsize=16)
plt.tight_layout(); plt.show()

print(f'Final training loss : {history["train_loss"][-1]:.4f}')
print(f'Best validation mIoU: {best_miou:.4f} at epoch {best_ep}')

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

In [ ]:
def predict(image_path, model, processor, device):
  image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
  orig_h, orig_w = image.shape[:2]

  encoded = processor(
      images  = image,
      return_tensors = 'pt'
  )
  pixel_values = encoded['pixel_values'].to(device)

  with torch.no_grad():
    outputs = model(pixel_values=pixel_values)

  upsampled = torch.nn.functional.interpolate(
      outputs.logits,
      size = (orig_h, orig_w),
      mode = 'bilinear',
      align_corners = False
  )
  pred_mask = upsampled.argmax(dim=1).squeeze(0).cpu().numpy()

  return image, pred_mask

In [ ]:
CLASS_COLORS = {0: (30, 30, 30), 1: (220, 50, 50), 2: (50, 200, 50), 3: (50, 50, 220)}

def mask_to_color(mask):
    h, w = mask.shape
    color_img = np.zeros((h, w, 3), dtype=np.uint8)
    for cid, col in CLASS_COLORS.items():
        color_img[mask == cid] = col
    return color_img

In [ ]:
def visualize_predictions(img_dir, mask_dir, model, processor, device, n=4):
    fnames = sorted(os.listdir(img_dir))[:n]
    fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
    if n == 1: axes = [axes]
    for i, fname in enumerate(fnames):
        img_path  = os.path.join(img_dir, fname)
        mask_path = os.path.join(mask_dir, fname)
        if not os.path.exists(mask_path):
            mask_path = os.path.join(mask_dir, os.path.splitext(fname)[0] + '.png')
        image, pred = predict(img_path, model, processor, device)
        gt   = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        overlay = (image.astype(np.float32) * 0.6 +
                   mask_to_color(pred).astype(np.float32) * 0.4).clip(0,255).astype(np.uint8)
        axes[i][0].imshow(image);              axes[i][0].axis('off')
        axes[i][1].imshow(mask_to_color(gt));  axes[i][1].axis('off')
        axes[i][2].imshow(mask_to_color(pred));axes[i][2].axis('off')
        axes[i][3].imshow(overlay);            axes[i][3].axis('off')
    for j, t in enumerate(['Original Image', 'Ground Truth', 'Prediction', 'Overlay']):
        axes[0][j].set_title(t, fontsize=12, fontweight='bold')
    patches = [mpatches.Patch(color=np.array(c)/255, label=ID2LABEL[cid])
               for cid, c in CLASS_COLORS.items() if cid < NUM_CLASSES]
    fig.legend(handles=patches, loc='lower center', ncol=NUM_CLASSES,
               fontsize=11, title='Classes', bbox_to_anchor=(0.5, -0.02))
    plt.suptitle('SegFormer Predictions', fontsize=16, y=1.01)
    plt.tight_layout(); plt.show()

In [ ]:
visualize_predictions(VALID_IMG_DIR, VALID_MASK_DIR, model, processor, device, n=4)

In [ ]:
model.eval()
metric_final = evaluate.load('mean_iou')

with torch.no_grad():
  for batch in valid_dataloader:
    pixel_values = batch['pixel_values'].to(device)
    labels = batch['labels'].to(device)

    outputs = model(pixel_values=pixel_values)
    logits = torch.nn.functional.interpolate(
        outputs.logits,
        size = labels.shape[-2:],
        mode = 'bilinear',
        align_corners = False
    )
    predicted = logits.argmax(dim=1)
    metric_final.add_batch(
        predictions = predicted.detach().cpu().numpy(),
        references = labels.detach().cpu().numpy()
    )

results = metric_final.compute(num_labels=NUM_CLASSES, ignore_index=255)
print('=' * 42)
print('  Final Evaluation Results')
print('=' * 42)
print(f'  Mean IoU      : {results["mean_iou"]:.4f}')
print(f'  Mean Accuracy : {results["mean_accuracy"]:.4f}')
print()
print('  Per-Class IoU:')
for cid, iou_val in enumerate(results['per_category_iou']):
    name = ID2LABEL.get(cid, f'class_{cid}')
    bar  = '█' * int(iou_val * 20)
    print(f'    {name:>15}: {iou_val:.4f}  {bar}')
print('=' * 42)

In [ ]:
SAVE_DIR = '/content/segformer_finetuned'

model.load_state_dict(torch.load(best_model_path, map_location=device))
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)

print(f' Model saved to: {SAVE_DIR}')
print('Files:')
for f in sorted(os.listdir(SAVE_DIR)):
  size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
  print(f'  {f:40s}  {size:.1f} MB')

```
LOAD_DIR = '/content/segformer_finetuned'
device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

loaded_processor = SegformerImageProcessor.from_pretrained(LOAD_DIR)
loaded_model     = SegformerForSemanticSegmentation.from_pretrained(LOAD_DIR)
loaded_model     = loaded_model.to(device)
loaded_model.eval()

print(' Model reloaded successfully')
print(f'   Classes: {loaded_model.config.id2label}')

# Quick test
test_img  = sorted(os.listdir(VALID_IMG_DIR))[0]
img, pred = predict(os.path.join(VALID_IMG_DIR, test_img), loaded_model, loaded_processor, device)
print(f'   Test image shape    : {img.shape}')
print(f'   Predicted mask shape: {pred.shape}')
print(f'   Unique classes found: {np.unique(pred).tolist()}')
print(' Inference working correctly!')
```